# Bricksmart デモデータ作成ノートブック

このノートブックは、**Databricks データアナリストワークショップ** で使用する架空のオンラインスーパー「**ブリックスマート (bricksmart)**」のサンプルデータを生成し、Unity Catalog 配下に書き込みます。

ワークショップ全体の流れと前提条件は、親ディレクトリの `README.md` の **「7. サンプルテーブルの作成」** を参照してください。

## 生成されるオブジェクト

| 名前 | 種別 | 内容 |
| --- | --- | --- |
| `users` | テーブル | 会員のマスタデータ（属性、地域、登録日など） |
| `products` | テーブル | 商品マスタ（カテゴリ、サブカテゴリ、価格など） |
| `transactions` | テーブル | 購買履歴（地域・年代・性別ごとの傾向を反映） |
| `orders_metric_view` | メトリクスビュー | Genie Spaces が利用する意味論レイヤー |

## このノートブックの流れ

1. ウィジェットで指定したカタログ／スキーマを準備
2. `users` / `products` のマスタデータを生成
3. 購買傾向を反映した `transactions` を生成
4. テーブル/カラムコメント、PII タグ、PK/FK 制約を付与
5. 列レベルマスキング・認定済みタグを適用（環境がサポートする場合）
6. メトリクスビューを作成

## 前提条件

- **Unity Catalog 有効** のワークスペース
- 実行ユーザーが対象カタログで **スキーマ作成権限** と **テーブル書き込み権限** を持つこと
- サーバーレス、または **DBR 15.4 以降** のクラスターでの実行を推奨（列レベルマスキング機能のため）


## 1. パラメータとカタログ・スキーマの準備

ウィジェットでカタログ名・スキーマ名・スキーマ再作成フラグを受け取り、入力検証のうえカタログとスキーマを準備します。

| ウィジェット | デフォルト | 説明 |
| --- | --- | --- |
| `catalog` | `data_analyst_workshop` | 使用する Unity Catalog のカタログ名（事前に作成済みのもの） |
| `schema` | `bricksmart` | 作成または使用するスキーマ名 |
| `recreate_schema` | `False` | `True` にすると既存スキーマを **削除して再作成** します（既存データは失われます） |


In [0]:
# Widgetsの作成
dbutils.widgets.text("catalog", "data_analyst_workshop", "カタログ")
dbutils.widgets.text("schema", "bricksmart", "スキーマ")
dbutils.widgets.dropdown("recreate_schema", "False", ["True", "False"], "スキーマを再作成")

# Widgetからの値の取得
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
recreate_schema = dbutils.widgets.get("recreate_schema") == "True"

In [0]:
print(f"catalog: {catalog}")
print(f"schema: {schema}")
print(f"recreate_schema: {recreate_schema}")

if not catalog:
    raise ValueError("存在するカタログ名を入力してください")
if not schema:
    raise ValueError("スキーマ名を入力してください")

In [0]:
# カタログを指定
spark.sql(f"USE CATALOG {catalog}")

# スキーマを再作成するかどうか
if recreate_schema:
    print(f"スキーマ {schema} を一度削除してから作成します")
    spark.sql(f"DROP SCHEMA IF EXISTS {schema} CASCADE;")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
else:
    print(f"スキーマ {schema} が存在しない場合は作成します (存在する場合は何もしません)")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

# スキーマを使用
spark.sql(f"USE SCHEMA {schema}")

## 2. マスタテーブルの生成（`users` / `products`）

ヘルパー関数で乱数ベースのユーザー名・商品名を生成し、`users` と `products` テーブルを作成します。

- **users**: 会員ごとの属性（年齢・性別・地域・メールアドレス・登録日 など）
- **products**: 食料品 / 日用品 / 家電 などのカテゴリとサブカテゴリ階層を持つ商品マスタ


In [0]:
from pyspark.sql.functions import udf, expr, when, col, lit, round, rand, greatest, least, date_format, dayofweek, concat
from pyspark.sql.types import StringType

import datetime
import random
import string

def generate_username():
    # 5文字のランダムな小文字アルファベットを生成
    part1 = ''.join(random.choices(string.ascii_lowercase, k=5))
    part2 = ''.join(random.choices(string.ascii_lowercase, k=5))
    
    # 形式 xxxxx.xxxxx で結合
    username = f"{part1}.{part2}"
    return username

def generate_productname():
    # 5文字のランダムな小文字アルファベットを生成
    part1 = ''.join(random.choices(string.ascii_lowercase, k=3))
    part2 = ''.join(random.choices(string.ascii_lowercase, k=3))
    part3 = ''.join(random.choices(string.ascii_lowercase, k=3))
    
    # 形式 xxx_xxx_xxx で結合
    productname = f"{part1}_{part2}_{part3}"
    return productname

generate_username_udf = udf(generate_username, StringType())
generate_productname_udf = udf(generate_productname, StringType())

# ユーザーデータの生成
def generate_users(num_users=10000):
    """
    ユーザーデータを生成し、指定された数のデータを返します。
    
    パラメータ:
    num_users (int): 生成するユーザーの数 (デフォルトは10000)
    
    戻り値:
    DataFrame: 生成されたユーザー情報を含むSpark DataFrame
    
    各ユーザーには以下のカラムが含まれます:
    - user_id: ユーザーID (1からnum_usersまでの範囲)
    - name: ランダムなユーザー名
    - age: ランダムな年齢 (一様分布: 18歳以上78歳未満)
    - gender: 男性48%、女性47%、その他2%、未回答3%
    - email: ユーザー名を基にしたメールアドレス
    - registration_date: 固定の日付 (2020年1月1日)
    - region: 東京40%、大阪25%、北海道20%、福岡10%、沖縄5%
    """
    return (
        spark.range(1, num_users + 1)
        .withColumnRenamed("id", "user_id")
        .withColumn("name", generate_username_udf())
        .withColumn("age", round(rand() * 60 + 18))
        .withColumn("rand_gender", rand())
        .withColumn(
            "gender",
            when(col("rand_gender") < 0.02, lit("その他")) # 2%
            .when(col("rand_gender") < 0.05, lit("未回答")) # 0.02 + 0.03 (3%)
            .when(col("rand_gender") < 0.53, lit("男性")) # 0.05 + 0.48 (48%)
            .otherwise(lit("女性")) # 残り47%
        )
        .withColumn("email", concat(col("name"), lit("@example.com")))
        .withColumn("registration_date", lit(datetime.date(2020, 1, 1)))
        .withColumn("rand_region", rand())
        .withColumn(
            "region",
            when(col("rand_region") < 0.40, lit("東京")) # 40%
            .when(col("rand_region") < 0.65, lit("大阪")) # 40% + 25% = 65%
            .when(col("rand_region") < 0.85, lit("北海道")) # 65% + 20% = 85%
            .when(col("rand_region") < 0.95, lit("福岡")) # 85% + 10% = 95%
            .otherwise(lit("沖縄")) # 残り5%
        )
        .drop("rand_gender", "rand_region")
    )

# 商品データの生成
def generate_products(num_products=100):
    """
    商品データを生成し、指定された数のデータを返します。
    
    パラメータ:
    num_products (int): 生成する商品の数 (デフォルトは100)
    
    戻り値:
    DataFrame: 生成された商品情報を含むSpark DataFrame
    
    各商品には以下のカラムが含まれます:
    - product_id: 商品ID (1からnum_productsまでの範囲)
    - product_name: ランダムな商品名
    - category: カテゴリ (食料品50%、日用品50%)
    - subcategory: サブカテゴリ
      食料品の場合: 野菜25%、果物25%、健康食品25%、肉類25%
      日用品の場合: キッチン用品25%、スポーツ・アウトドア用品25%、医薬品25%、冷暖房器具25%
    - price: 商品価格 (100円以上1100円未満の範囲)
    - stock_quantity: 在庫数 (1以上101未満の範囲)
    - cost_price: 仕入れ価格 (販売価格の70%)
    """
    return (
        spark.range(1, num_products + 1)
        .withColumnRenamed("id", "product_id")
        .withColumn("product_name", generate_productname_udf())
        .withColumn("rand_category", rand())
        .withColumn(
            "category",
            when(col("rand_category") < 0.5, lit("食料品")).otherwise(lit("日用品"))
        )
        .withColumn("rand_subcategory", rand())
        .withColumn(
            "subcategory",
            when(
                col("category") == "食料品",
                when(col("rand_subcategory") < 0.25, lit("野菜"))
                .when(col("rand_subcategory") < 0.50, lit("果物"))
                .when(col("rand_subcategory") < 0.75, lit("健康食品"))
                .otherwise(lit("肉類"))
            ).otherwise(
                when(col("rand_subcategory") < 0.25, lit("キッチン用品"))
                .when(col("rand_subcategory") < 0.50, lit("スポーツ・アウトドア用品"))
                .when(col("rand_subcategory") < 0.75, lit("医薬品"))
                .otherwise(lit("冷暖房器具"))
            )
        )
        .withColumn("price", round(rand() * 1000 + 100, 2))
        .withColumn("stock_quantity", round(rand() * 100 + 1))
        .withColumn("cost_price", round(col("price") * 0.7, 2))
        .drop("rand_category", "rand_subcategory")
    )

users = generate_users()
products = generate_products()

display(users.limit(5))
display(products.limit(5))

In [0]:
users.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("users")
products.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("products")

## 3. 購買履歴データの生成（`transactions`）

各ユーザーの **地域 × 年代 × 性別** に応じた購買傾向（傾向スコア = 重み付け）を定義し、確率的に取引レコードを生成します。例:

- 東京は食料品の購入比率が高め
- 北海道は冷暖房器具の購入が多い
- 若年層はスナック・スイーツを好む傾向

これにより、後続の集計クエリや AI/BI ダッシュボードで意味のある可視化結果が得られます。

> 一旦 `transactions_temp` に書き込み、`DEEP CLONE` でメインテーブル `transactions` を作成 → 一時テーブルを削除する流れを取っています。これは **Unity Catalog のリネージグラフを直感的に分かりやすくする** ための工夫です。


In [0]:
# 購買行動に関する傾向スコア（重み）の設定
conditions = [
    # ---------- 地域ごとの傾向 ----------
    # 東京: 食生活において多様性を求める傾向があり、食料品の購入量が増える
    ((col("region") == "東京") & (col("category") == "食料品"), 1),

    # 大阪: 実用的な日用品の購入を好む
    ((col("region") == "大阪") & (col("category") == "日用品"), 1),

    # 福岡: 健康志向の高い野菜を多く購入
    ((col("region") == "福岡") & (col("subcategory") == "野菜"), 1),

    # 北海道: 寒冷地のため冷暖房器具の購入量が増える
    ((col("region") == "北海道") & (col("subcategory") == "冷暖房器具"), 2),

    # 沖縄: 地元の果物への関心が高い。さらに温暖な気候のため冷暖房器具の購入量が増える
    ((col("region") == "沖縄") & (col("subcategory") == "果物"), 1),
    ((col("region") == "沖縄") & (col("subcategory") == "冷暖房器具"), 1),

    # ---------- 性別ごとの傾向 ----------
    # 女性: 食料品や日用品、特にキッチン用品を多く購入
    ((col("gender") == "女性") & (col("category") == "食料品"), 1),
    ((col("gender") == "女性") & (col("category") == "日用品"), 1),
    ((col("gender") == "女性") & (col("subcategory") == "キッチン用品"), 1),

    # 男性: スポーツやアウトドア関連の商品に関心が高い。さらに肉類を好む傾向が強い
    ((col("gender") == "男性") & (col("category") == "スポーツ・アウトドア用品"), 2),
    ((col("gender") == "男性") & (col("subcategory") == "肉類"), 1),

    # ---------- 年齢層ごとの傾向 ----------
    # 若年層 (18〜34歳): 果物、肉類、スポーツ・アウトドア用品に関心が高い
    ((col("age") < 35) & (col("subcategory") == "果物"), 1),
    ((col("age") < 35) & (col("subcategory") == "肉類"), 2),
    ((col("age") < 35) & (col("subcategory") == "スポーツ・アウトドア用品"), 2),

    # 中年層 (35〜54歳): 健康志向が高まり野菜の購入量が増える。肉類もそれなりに購入。医薬品の購入量も増える
    ((col("age") >= 35) & (col("age") < 55) & (col("subcategory") == "野菜"), 1),
    ((col("age") >= 35) & (col("age") < 55) & (col("subcategory") == "肉類"), 1),
    ((col("age") >= 35) & (col("age") < 55) & (col("subcategory") == "医薬品"), 1),

    # シニア層 (55歳以上): 果物と野菜、医薬品の購入量が増える
    ((col("age") >= 55) & (col("subcategory") == "果物"), 2),
    ((col("age") >= 55) & (col("subcategory") == "野菜"), 2),
    ((col("age") >= 55) & (col("subcategory") == "医薬品"), 2),

    # ---------- 組み合わせによる傾向 ----------
    # 東京の若年層: 消費行動が旺盛で全体的な購入量が多い
    ((col("region") == "東京") & (col("age") < 35), 1),

    # 大阪の中年層: 家庭を持ち、食料品の購入量が増える
    ((col("region") == "大阪") & (col("age") >= 35) & (col("age") < 55) & (col("category") == "食料品"), 2),

    # 北海道の若年層: アウトドア活動に関連する日用品を購入する
    ((col("region") == "北海道") & (col("age") < 35) & (col("category") == "日用品"), 1),

    # 沖縄のシニア層は地元の伝統食に高い関心を持つ
    ((col("region") == "沖縄") & (col("age") >= 55) & (col("category") == "食料品"), 2),
]

# トランザクションデータの生成
def generate_transactions(users, products, num_transactions=1000000):
    """
    トランザクションデータを生成し、指定された数のデータを返します。

    パラメータ:
    users (DataFrame): ユーザーデータを含むSpark DataFrame
    products (DataFrame): 商品データを含むSpark DataFrame
    num_transactions (int): 生成するトランザクションの数 (デフォルトは1000000)

    戻り値:
    DataFrame: 生成されたトランザクション情報を含むSpark DataFrame

    各トランザクションには以下のカラムが含まれます:
    - transaction_id: トランザクションID (1からnum_transactionsまでの範囲)
    - user_id: ユーザーID (1から登録ユーザー数までの範囲)
    - product_id: 商品ID (1から登録商品数までの範囲)
    - quantity: 購入数量 (1以上6以下の整数、傾向スコアによって調整)
    - store_id: 店舗ID (1以上11以下の整数)
    - transaction_date: 取引日 (2023年1月1日から2024年1月1日までの範囲)
        - 8月と12月は10%の確率で特定の日付を選択
        - 週末は10%の確率で特定の日付を選択
    - transaction_price: 取引金額 (quantity * price)

    傾向スコア:
    ユーザーの属性や商品カテゴリに基づいて購入数量を調整します。
    最終的な数量は0以上の範囲に収まるように調整されます。
    """
    transactions = (
        spark.range(1, num_transactions + 1).withColumnRenamed("id", "transaction_id")
        .withColumn("user_id", expr(f"floor(rand() * {users.count()}) + 1"))
        .withColumn("product_id", expr(f"floor(rand() * {products.count()}) + 1"))
        .withColumn("quantity", round(rand() * 5 + 1))
        .withColumn("store_id", round(rand() * 10 + 1))
        .withColumn("random_date", expr("date_add(date('2024-01-01'), -CAST(rand() * 365 AS INTEGER))"))
        .withColumn("month", date_format("random_date", "M").cast("int"))
        .withColumn("is_weekend", dayofweek("random_date").isin([1, 7]))
        .withColumn("transaction_date", 
            when((rand() < 0.1) & ((expr("month") == 8) | (expr("month") == 12)), expr("random_date"))
            .when((rand() < 0.1) & expr("is_weekend"), expr("random_date"))
            .otherwise(expr("date_add(date('2024-01-01'), -CAST(rand() * 365 AS INTEGER))"))
        )
        .drop("random_date", "month", "is_weekend")
    )

    # 傾向スコアに基づいて購入数量を調整
    adjusted_transaction = transactions.join(users, "user_id").join(products.select("product_id", "price", "category", "subcategory"), "product_id")
    for condition, adjustment in conditions:
        adjusted_transaction = adjusted_transaction.withColumn("quantity", when(condition, col("quantity") + adjustment).otherwise(col("quantity")))
    adjusted_transaction = adjusted_transaction.withColumn("quantity", greatest(lit(0), "quantity"))
    adjusted_transaction = adjusted_transaction.withColumn("transaction_price", col("quantity") * col("price"))

    # 調整済みトランザクションデータを返却
    return adjusted_transaction.select("transaction_id", "user_id", "product_id", "quantity", "transaction_price", "transaction_date", "store_id")


users = spark.table("users")
products = spark.table("products")
transactions = generate_transactions(users, products)
# 結果の表示（データフレームのサイズによっては表示が重くなる可能性があるため、小さなサンプルで表示）
display(transactions.limit(5))


Note: トランザクションテーブルへの書き込みについて、リネージの流れを直感的に分かりやすいものにするために一旦一時テーブルに書き込み、DEEP CLONEを使用してメインテーブルを作成する。


In [0]:
transactions.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("transactions_temp")


In [0]:
spark.sql("DROP TABLE IF EXISTS transactions")
spark.sql("CREATE TABLE transactions DEEP CLONE transactions_temp")


In [0]:
spark.sql("DROP TABLE transactions_temp")


## 4. メタデータの整備（コメント・タグ・PK/FK）

Unity Catalog 上のテーブル/カラムにコメントや制約を追加し、データの意味とリレーションを明示します。

- **テーブル/カラムコメント**: Genie Spaces やカタログエクスプローラ上で、各テーブル・カラムの意味が解釈しやすくなる
- **PII タグ**（コメントアウト済、必要に応じて有効化）: 個人情報を含むカラムをマーキング
- **プライマリキー / 外部キー**: 論理的なリレーションを宣言（FK は `NOT ENFORCED` で物理的な強制はなし）


In [0]:
%sql
ALTER TABLE users ALTER COLUMN user_id COMMENT "ユーザーID";
ALTER TABLE users ALTER COLUMN name COMMENT "氏名";
ALTER TABLE users ALTER COLUMN age COMMENT "年齢: 0以上";
ALTER TABLE users ALTER COLUMN gender COMMENT "性別: 例) 男性, 女性, 未回答, その他";
ALTER TABLE users ALTER COLUMN email COMMENT "メールアドレス";
ALTER TABLE users ALTER COLUMN registration_date COMMENT "登録日";
ALTER TABLE users ALTER COLUMN region COMMENT "地域: 例) 東京, 大阪, 北海道";
COMMENT ON TABLE users IS '**users テーブル**\nオンラインスーパー「ブリックスマート」に登録されているユーザー情報を保持するテーブルです。\n- ユーザーの基本情報（氏名、年齢、性別、地域など）や連絡先（メールアドレス）を管理\n- ユーザーのセグメンテーションや嗜好分析、マーケティング効果測定などに活用できます';

ALTER TABLE transactions ALTER COLUMN transaction_id COMMENT "トランザクションID";
ALTER TABLE transactions ALTER COLUMN user_id COMMENT "ユーザーID: usersテーブルのuser_idとリンクする外部キー";
ALTER TABLE transactions ALTER COLUMN transaction_date COMMENT "購入日";
ALTER TABLE transactions ALTER COLUMN product_id COMMENT "商品ID: productsテーブルのproduct_idとリンクする外部キー";
ALTER TABLE transactions ALTER COLUMN quantity COMMENT "購入数量: 1以上";
ALTER TABLE transactions ALTER COLUMN transaction_price COMMENT "購入時価格: 0以上, transactions.quantity * products.price で計算";
ALTER TABLE transactions ALTER COLUMN store_id COMMENT "店舗ID";
COMMENT ON TABLE transactions IS '**transactions テーブル**\nオンラインスーパー「ブリックスマート」で行われた販売取引（購入履歴）の情報を管理するテーブルです。\n- ユーザーIDや商品IDなど他テーブルと関連付けしつつ、購入日や価格、数量などを保持\n- 販売動向の分析、ユーザーの購買行動追跡、在庫・マーケティング戦略の最適化に役立ちます';

ALTER TABLE products ALTER COLUMN product_id COMMENT "商品ID";
ALTER TABLE products ALTER COLUMN product_name COMMENT "商品名";
ALTER TABLE products ALTER COLUMN category COMMENT "カテゴリー: 例) 食料品, 日用品";
ALTER TABLE products ALTER COLUMN subcategory COMMENT "サブカテゴリー: 例) 野菜, 洗剤";
ALTER TABLE products ALTER COLUMN price COMMENT "販売価格: 0以上";
ALTER TABLE products ALTER COLUMN stock_quantity COMMENT "在庫数量";
ALTER TABLE products ALTER COLUMN cost_price COMMENT "仕入れ価格";
COMMENT ON TABLE products IS '**products テーブル**\nオンラインスーパー「ブリックスマート」で取り扱う商品の情報を管理するテーブルです。\n- 商品名、カテゴリー・サブカテゴリー、価格、在庫数、原価などを保持\n- 在庫管理、価格分析、商品分類や商品のパフォーマンス分析に活用できます';



In [0]:
%sql
-- ALTER TABLE users ALTER COLUMN name SET TAGS ('pii_name');
-- ALTER TABLE users ALTER COLUMN email SET TAGS ('pii_email');


In [0]:
%sql
ALTER TABLE users ALTER COLUMN user_id SET NOT NULL;
ALTER TABLE transactions ALTER COLUMN transaction_id SET NOT NULL;
ALTER TABLE products ALTER COLUMN product_id SET NOT NULL;

ALTER TABLE users ADD CONSTRAINT users_pk PRIMARY KEY (user_id);
ALTER TABLE transactions ADD CONSTRAINT transactions_pk PRIMARY KEY (transaction_id);
ALTER TABLE products ADD CONSTRAINT products_pk PRIMARY KEY (product_id);

ALTER TABLE transactions ADD CONSTRAINT transactions_users_fk FOREIGN KEY (user_id) REFERENCES users (user_id) NOT ENFORCED;
ALTER TABLE transactions ADD CONSTRAINT transactions_products_fk FOREIGN KEY (product_id) REFERENCES products (product_id) NOT ENFORCED;


## 5. ガバナンス機能の適用（マスキング・認定済みタグ）

データ取り扱いに関するガバナンス機能を適用します。環境（DBR バージョンや機能対応状況）によっては失敗してもスキップされ、ノートブックは続行します。

- **列レベルマスキング**: `email` カラムを `admins` グループメンバー以外には `***@example.com` でマスク
- **認定済みタグ (`system.Certified`)**: 信頼できるデータセットであることをマーキング


In [0]:
try:
    # マスキング関数の作成
    spark.sql("""
    CREATE FUNCTION IF NOT EXISTS mask_email(email STRING) 
    RETURN CASE WHEN is_member('admins') THEN email ELSE '***@example.com' END
    """)
    
    # usersテーブルにマスキングを適用
    spark.sql("""
    ALTER TABLE users ALTER COLUMN email SET MASK mask_email
    """)
    
    print("列レベルマスキングの適用が完了しました。")
    
except Exception as e:
    print(f"列レベルマスキングの適用中にエラーが発生しました: {str(e)}")
    print("このエラーはDBR 15.4より前のバージョンで実行している場合に発生する可能性があります。")


In [0]:
certified_tag = 'system.Certified'

try:
    spark.sql(f"ALTER TABLE users SET TAGS ('{certified_tag}')")
    spark.sql(f"ALTER TABLE transactions SET TAGS ('{certified_tag}')")
    spark.sql(f"ALTER TABLE products SET TAGS ('{certified_tag}')")
    print(f"認定済みタグ '{certified_tag}' の追加が完了しました。")

except Exception as e:
    print(f"認定済みタグ '{certified_tag}' の追加中にエラーが発生しました: {str(e)}")
    print("このエラーはタグ機能に対応していないワークスペースで実行した場合に発生する可能性があります。")

## 6. Genie Spaces 用のメトリクスビューを作成

Genie Spaces で自然言語クエリを実行する際の **意味論レイヤー** となるメトリクスビュー (`orders_metric_view`) を作成します。

- `transactions` をファクト、`users` / `products` をディメンションとして JOIN
- ディメンション（購入日・地域・性別 など）と メジャー（売上 など）を YAML で宣言的に定義

このビューを Genie Spaces のデータソースとして指定することで、ビジネス文脈に沿った日本語の問い合わせに高精度で応答できるようになります。


In [0]:
sql = f"""
CREATE OR REPLACE VIEW orders_metric_view
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: {catalog}.{schema}.transactions
  joins:
    - name: products
      source: {catalog}.{schema}.products
      on: source.product_id = products.product_id
    - name: users
      source: {catalog}.{schema}.users
      on: source.user_id = users.user_id
  comment: 購入履歴データのメトリクスビュー
  dimensions:
    - name: order_date
      expr: transaction_date
      display_name: 購入日
      comment: 顧客が商品を購入した日付
      format: 
        type: date
        date_format: year_month_day
    - name: user_id
      expr: user_id
      display_name: ユーザーID
      comment: ユーザーID。usersテーブルのuser_idとリンクする外部キー
    - name: product_id
      expr: products.product_id
      display_name: 商品ID
      comment: 商品ID。productsテーブルのproduct_idとリンクする外部キー
    - name: category
      expr: products.category
      display_name: カテゴリ
      comment: 商品のカテゴリー。例) 食料品, 日用品
    - name: subcategory
      expr: products.subcategory
      display_name: サブカテゴリ
      comment: 商品のサブカテゴリー。例) 野菜, 洗剤
    - name: age
      expr: users.age
      display_name: 年齢
      comment: 顧客の年齢。0以上
    - name: age_group
      expr: CASE WHEN users.age < 35 THEN '若年層' WHEN users.age < 55 THEN '中年層' ELSE 'シニア層' END
      display_name: 年齢層
      comment: 顧客の年齢層。例) 若年層, 中年層, シニア層
    - name: gender
      expr: users.gender
      display_name: 性別
      comment: 顧客の性別。) 男性, 女性, 未回答, その他
    - name: region
      expr: users.region
      display_name: 地域
      comment: 顧客の居住地域。) 東京, 大阪, 北海道
  measures:
    - name: total_purchace_amount
      expr: SUM(transaction_price)
      display_name: 合計購入金額
      format:
        type: currency
        currency_code: JPY
      synonyms:
        - Total Purchase Amount
        - 購入金額合計
      comment: 購入された商品の合計金額
    - name: total_purchase_count
      expr: COUNT(transaction_id)
      display_name: 合計購入回数
      synonyms:
        - Total Purchase Count
        - 延べ人数
      comment: 合計の購入回数。または購入者の延べ人数
    - name: total_unique_users
      expr: COUNT(DISTINCT user_id)
      display_name: 客数
      comment: 購入した顧客数で、重複を排除したもの。ユニークユーザー数
      synonyms:
        - Total Unique Users
        - ユニークユーザー数

    - name: unit_price
      expr: SUM(transaction_price) / COUNT(transaction_id)
      display_name: 客単価
      format:
        type: currency
        currency_code: JPY
      comment: 1購入あたりの購入金額。合計金額を購入回数で割った値。
      synonyms:
        - 1購入あたり金額
    - name: frequency
      expr: COUNT(transaction_id) / COUNT(DISTINCT user_id)
      display_name: 購入頻度
      comment: 1ユーザーあたりの購入回数。購入回数を客数(ユニークユーザー数)で割った値
      synonyms:
        - 1人あたり購入回数
$$
"""

spark.sql(sql)

In [ ]:
%sql
SELECT 
  MEASURE(`total_purchace_amount`) AS `total_purchace_amount`, 
  MEASURE(`total_purchase_count`) AS `total_purchase_count`, 
  MEASURE(`total_unique_users`) AS `total_unique_users`, 
  MEASURE(`unit_price`) AS `unit_price`, 
  MEASURE(`frequency`) AS `frequency` 
FROM orders_metric_view LIMIT 10;

お疲れ様でした。エラーなく実行ができたことを確認したら、次はREADME.mdの8. サンプルテーブルの確認を進めましょう。